In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline


## 1. データを作る

In [ ]:
df = pd.DataFrame({
    'day':   [1,2,3,4,5,6,7,8,9,10,11,12],
    'temp':  [14,17,12,15,19,22,20,25,24,18,22,17],
    'sales': [54,79,46,81,97,125,100,147,131,102,111,99]
})
df

このデータでは，説明変数 `temp` から目的変数 `sales` を予測する．

線形モデルは次の形．

$$\hat{y}=wx+b$$

ここで，$w$ は傾き（重み），$b$ は切片（バイアス）を表す．

## 2. 散布図で確認する

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(df['temp'], df['sales'])
plt.xlabel('temperature')
plt.ylabel('sales')
plt.title('Temperature and Sales')
plt.grid(True)
plt.show()

## 3. 線形（回帰）モデルのトレーニング：学習

最小二乗法では，実測値 $y$ と予測値 $\hat{y}$ の差を二乗し，その合計が小さくなるように $w$ と $b$ を決めます。

$$\sum_i (y_i - \hat{y}_i)^2$$

In [ ]:
X = df[['temp']]
y = df['sales']

model = LinearRegression()
model.fit(X, y)

w = model.coef_[0]
b = model.intercept_
print(f'回帰式: sales = {w:.2f} * temp + {b:.2f}')

## 4. 回帰直線を描く

In [ ]:
x_line = np.linspace(df['temp'].min()-1, df['temp'].max()+1, 100).reshape(-1, 1)
y_line = model.predict(x_line)

plt.figure(figsize=(6,4))
plt.scatter(df['temp'], df['sales'], label='data')
plt.plot(x_line, y_line, label='linear regression')
plt.xlabel('temperature')
plt.ylabel('sales')
plt.title('Linear Regression')
plt.legend()
plt.grid(True)
plt.show()

## 5. 損失関数（二乗誤差）を確認する

RMSEは，単にMSE：二乗誤差の平方根をとったもの．オーダーがもとの変数とあう．$R^2$はMSEを分散で（「平均との差の二乗」を足し上げてnで割る）割ったものを1から引いて求める：1に近いほどモデルの予測性能が高い．

In [ ]:
df_eval = df.copy()
df_eval['pred'] = model.predict(X)
df_eval['error'] = df_eval['sales'] - df_eval['pred']
df_eval['squared_error'] = df_eval['error'] ** 2
display(df_eval)

mse = mean_squared_error(y, df_eval['pred'])
rmse = np.sqrt(mse)
r2 = r2_score(y, df_eval['pred'])

print(f'MSE: {mse:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'R^2: {r2:.3f}')

## 6. 訓練データとテストデータに分ける

学習に使ったデータで性能を見るだけでは不十分．未知のデータでどれくらい当たるか？つまり **汎化性能** を見る必要がある：我々は，明日の予想最高気温がわかったときにアイスがどれくらい売れるか知りたいのだ．

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=0
)

model_train = LinearRegression()
model_train.fit(X_train, y_train)

pred_train = model_train.predict(X_train)
pred_test = model_train.predict(X_test)

print('訓練データ')
print(f'RMSE: {np.sqrt(mean_squared_error(y_train, pred_train)):.2f}')
print(f'R^2: {r2_score(y_train, pred_train):.3f}')

print('\nテストデータ')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, pred_test)):.2f}')
print(f'R^2: {r2_score(y_test, pred_test):.3f}')

## 7. 訓練データとテストデータを図で見る

In [ ]:
x_line = np.linspace(df['temp'].min()-1, df['temp'].max()+1, 100).reshape(-1, 1)
y_line = model_train.predict(x_line)

plt.figure(figsize=(6,4))
plt.scatter(X_train['temp'], y_train, label='train')
plt.scatter(X_test['temp'], y_test, label='test')
plt.plot(x_line, y_line, label='model trained on train data')
plt.xlabel('temperature')
plt.ylabel('sales')
plt.title('Train/Test Split')
plt.legend()
plt.grid(True)
plt.show()

## 8. 過学習を体験する：多項式回帰

線形モデルは単純で良いが，モデルを複雑にした方が訓練データにはよく合う．

たとえば3次式なら：

$$\hat{y}=a_3x^3+a_2x^2+a_1x+b$$

次数を上げるほど，訓練データにはfitする．しかし，調子に乗ってモデルを複雑にしすぎる（次数を上げたり変数を増やしまくる）と，未知データへの予測が悪くなることがある．これが過学習．

In [ ]:
degrees = [1, 2, 3, 5, 9]
results = []

for degree in degrees:
    poly_model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    poly_model.fit(X_train, y_train)

    pred_train = poly_model.predict(X_train)
    pred_test = poly_model.predict(X_test)

    results.append({
        'degree': degree,
        'train_RMSE': np.sqrt(mean_squared_error(y_train, pred_train)),
        'test_RMSE': np.sqrt(mean_squared_error(y_test, pred_test)),
        'train_R2': r2_score(y_train, pred_train),
        'test_R2': r2_score(y_test, pred_test)
    })

pd.DataFrame(results)

## 9. モデルの複雑さと過学習を図で見る

In [ ]:
x_line = np.linspace(df['temp'].min()-1, df['temp'].max()+1, 400).reshape(-1, 1)

for degree in degrees:
    poly_model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    poly_model.fit(X_train, y_train)
    y_line = poly_model.predict(x_line)

    plt.figure(figsize=(6,4))
    plt.scatter(X_train['temp'], y_train, label='train')
    plt.scatter(X_test['temp'], y_test, label='test')
    plt.plot(x_line, y_line, label=f'degree={degree}')
    plt.xlabel('temperature')
    plt.ylabel('sales')
    plt.title(f'Polynomial Regression: degree={degree}')
    plt.legend()
    plt.grid(True)
    plt.show()

## 10. 演習

1. 訓練データにday13, day14, ... day20まで自由に数値を足してみて，散布図や回帰式，MSEなどの評価指標がどう変わるか観察せよ．
※もとの1～7セル（出来る人は8，9セルも）をコピーして，「10．演習」の下に貼ってから行うこと．
2. `random_state=0` を 1, 2, 3 に変えると，テスト性能はどう変わるか？
3. `test_size=0.33` を 0.25 や 0.5 に変えるとどうなるか？
4. 多項式の次数を `1,2,3,5,9,11` に変えると，訓練MSEとテストMSEはどう変わるか？
5. このデータ数で9次式や11次式を使うことは妥当か？
